In [ ]:
import os

# Set these BEFORE importing TensorFlow/Keras.
# For reliable offline use, replace with a fixed path, e.g. r"D:\ml_project"
BASE_DIR = os.getcwd()
os.environ["KERAS_HOME"] = os.path.join(BASE_DIR, "my_keras_cache")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# --- 1. Prepare and Preprocess the Dataset (CIFAR-10) ---
print("Loading CIFAR-10 (downloads on first run, then loads from cache)...")
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = tf.keras.datasets.cifar10.load_data()

num_classes = 10
print(f"Number of classes in dataset: {num_classes}")

def preprocess_image_pipeline(images, labels):
    images = tf.cast(images, tf.float32)
    # Upscale 32x32 -> 96x96 (values stay in 0-255)
    images = tf.image.resize(images, (96, 96))
    # Scale to [-1, 1], which is what MobileNetV2 was trained with
    images = preprocess_input(images)
    return images, labels

batch_size = 32

ds_train = tf.data.Dataset.from_tensor_slices((x_train_raw, y_train_raw))
ds_train = ds_train.shuffle(5000).batch(batch_size)
ds_train = ds_train.map(preprocess_image_pipeline, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

ds_test = tf.data.Dataset.from_tensor_slices((x_test_raw, y_test_raw))
ds_test = ds_test.batch(batch_size)
ds_test = ds_test.map(preprocess_image_pipeline, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)


# --- 2. Load Pre-trained CNN Model (Step a) ---
print("\nLoading pre-trained MobileNetV2 base network...")
base_model = MobileNetV2(
    weights="imagenet",   # downloads once, then loads from my_keras_cache/models/
    include_top=False,
    input_shape=(96, 96, 3),
)


# --- 3. Freeze Parameters in Lower Layers (Step b) ---
print("Freezing base model weights...")
base_model.trainable = False


# --- 4. Add Custom Trainable Classifier (Step c) ---
print("Building the custom classifier...")
inputs = tf.keras.Input(shape=(96, 96, 3), name="Input_Image")

x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="Global_Average_Pooling")(x)
x = tf.keras.layers.Dense(256, activation="relu", name="Dense_Trainable_1")(x)
x = tf.keras.layers.Dropout(0.3, name="Dropout_Regularization")(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="Output_Predictions")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Transfer_Learning_Classifier")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


# --- 5. Train Classifier Layers (Step d) ---
epochs_initial = 2
print(f"\nTraining custom classifier layers for {epochs_initial} epochs...")
history = model.fit(
    ds_train,
    epochs=epochs_initial,
    validation_data=ds_test,
)


# --- 6. Fine-Tuning: Unfreeze Top Layers & Train (Step e) ---
print("\n[Fine-Tuning Process Started]")
base_model.trainable = True

# Keep lower layers frozen, unfreeze from layer 100 onwards
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile with a lower learning rate (required after changing trainable flags)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
print(f"Base Model Layers: {len(base_model.layers)} | "
      f"Unfrozen Layers: {len(base_model.layers) - fine_tune_at}")

epochs_fine = 2
total_epochs = epochs_initial + epochs_fine

print(f"\nFine-tuning for an additional {epochs_fine} epochs...")
history_fine = model.fit(
    ds_train,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1,
    validation_data=ds_test,
)
print("\nTraining workflow execution complete.")

Loading built-in dataset safely from Keras local repository...
A local file was found, but it seems to be incomplete or outdated because the auto file hash does not match the original value of 6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce so we will re-download the data.


 38576128/170498071 ━━━━━━━━━━━━━━━━━━━━ 1:02:00 28us/step